In [1]:
import cv2
from cv2 import aruco
import numpy as np
import msgpack as mp
import msgpack_numpy as mpn
import os
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from joblib import Parallel, delayed

### Path defs

In [5]:
from pathlib import Path

# Path defs
project_root = Path.cwd().parents[2]
data_root = "data"
recording_type = 'calibration'
camera_type = 'dual_160'
calib_folder_name = "dual_cam_calibration_checker_sz_30mm"

calib_data_folder = os.path.join(
    project_root, data_root, recording_type, camera_type, calib_folder_name
)
cam_ov9281_data = os.path.join(calib_data_folder, "cam1_ov9281.msgpack")
cam_ov9281_meta = os.path.join(calib_data_folder, "cam1_timestamp.msgpack")

cam_imx219_data = os.path.join(calib_data_folder, "cam0_imx219.msgpack")
cam_imx219_meta = os.path.join(calib_data_folder, "cam0_timestamp.msgpack")
os.path.exists(cam_imx219_meta)

True

### Parse metadata

In [6]:
def get_metadata(metaf):
    f = open(metaf, 'rb')
    _ = np.array(list(mp.Unpacker(f, object_hook=mpn.decode)))
    sync, timestamps = _[:,0], _[:,1]
    sync = sync.astype(int).astype(bool)
    timestamp_dt = timestamps.astype('datetime64[us]')
    return sync, timestamp_dt

sync_cam0, timestamp_cam0 = get_metadata(cam_imx219_meta)
sync_cam1, timestamp_cam1 = get_metadata(cam_ov9281_meta)

In [12]:
duration = timestamp_cam0[-1] - timestamp_cam0[0]
# duration in seconds
duration_s = duration.astype('timedelta64[s]').astype(float)
print(f"Duration of recording: {duration_s:.2f} seconds")

Duration of recording: 142.00 seconds


### Get video unpacker file

In [56]:
def get_video_unpacker(vidf):
    _video_file = open(vidf, "rb")
    _video_data = mp.Unpacker(_video_file, object_hook=mpn.decode)
    return _video_data

cam0_upak = get_video_unpacker(cam_imx219_data)
cam1_upak = get_video_unpacker(cam_ov9281_data)